# 1.3 Model Selection

In [4]:
#__ Import: Numerical __
import numpy as np

#__ Import: Topophone2 __
from topophone2.topophone import decompose
from topophone2.inversion import MarginalStats
from topophone2.utils.read import read_txt

#__ Import: Plotting __
import matplotlib.pyplot as plt
import matplotlib.tri as tri
import matplotlib.ticker as ticker
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import cmcrameri.cm as cmc

#__ Set up: Plotting __
theme = ['#DC4424', '#74A466', '#24448C', '#F8BC5C']
small, large = 8, 10
plt.rcParams["font.sans-serif"] = ['Arial']

#__ Set up: Random seed __
np.random.seed(2026)

#__ Set up: File paths __
data_dir  = '../data/TRIAD_15DB'
model_dir = '../models/ALT'

#__ Disable: Warnings __
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

### 1.3.1 Create alternative characteristic function

In [5]:
#__ Apparent velocity beamwidth only __
def cf_radial2D(
    t   : np.ndarray,
    m_s : np.ndarray
):
    r"""

    log(cf) = i (t.xi) - (1/2) sigma^2 (t.xi)^2 / (xi.xi)

    Input
    -----
    `t` : `numpy.ndarray`
        Characteristic function argument, omega * R
        (frequency, sensor pair, direction)

    `m_s` : `numpy.ndarray`
        Model with mean slowness (X,Y) and radial beamwidth parameters
        (model, parameter)

    Output
    ------
    `cf` : `numpy.ndarray`
        Characteristic function
        (model, frequency, sensor pair)

    `dcf` : `numpy.ndarray`
        Gradient of characteristic function
        (model, frequency, sensor pair, parameter)
    """

    #__ np.einsum subscripts:
    #__ [i] model
    #__ [j] frequency
    #__ [k] sensor pair
    #__ [l],[p] direction

    #__ Unpack model vector __
    xi    = m_s[:, :2]  # (i,l)
    sigma = m_s[:, 2]   # (i,)

    #__ Invariants __
    u  = np.einsum('jkl,il->ijk', t, xi, optimize=True)  # (i,j,k)  t.xi
    s  = np.einsum('il,il->i', xi, xi, optimize=True)    # (i,)     xi.xi
    s3 = s[:, None, None]                                # (i,1,1)
    g  = u / s3                                          # (i,j,k)
    v2 = (sigma**2)[:, None, None]                       # (i,1,1)

    #__ Calculate characteristic function __
    L  = 1j * u
    Q  = -0.5 * v2 * u * g
    cf = np.exp(L + Q)

    #__ Calculate gradients __
    #__ d/dxi  = i t - sigma^2 (u/s) [ t - (u/s) xi ]
    #__ d/dsig = -sigma u^2 / s
    t_perp = t[None, ...] - g[..., None] * xi[:, None, None, :]     # (i,j,k,l)
    dlog          = np.empty(cf.shape + (m_s.shape[1],), dtype=cf.dtype)
    dlog[...]     = 0.0
    dlog[..., :2] = 1j * t[None, ...] - (v2 * g)[..., None] * t_perp
    dlog[..., 2]  = -sigma[:, None, None] * u * g
    dcf           = cf[..., None] * dlog

    #__ Return output __
    return cf, dcf

#__ `custom_cf` input __
custom_cf = {'radial2D':(
    3,           # Number of model parameters
    cf_radial2D  # Characteristic function
)}

### 1.3.2 Calculate posterior pdf under alternative parameterisation

Read in data from Example 1.1...

In [6]:
#__ Read: Signal + noise __
out = read_txt(f"{data_dir}/TRIAD_FULL_15DB_topophone2.txt")
freq_Hz   = out[0]  # Frequency vector         [Hz]        (frequency)
x         = out[1]  # Receiver position        [m]         (receiver, direction)
Gamma_obs = out[2]  # Array-covariance matrix  [uPa2.Hz-1] (frequency, receiver, receiver)
N_obs     = out[3]  # Independent observations
freq      = 2 * np.pi * freq_Hz  # Angular frequency vector [rad.s-1] (frequency)

#__ Read: Noise only __
out = read_txt(f"{data_dir}/TRIAD_NOISE_15DB_topophone2.txt")
Gamma_obs_noise = out[2]  # Array-covariance matrix  [uPa2.Hz-1] (frequency, receiver, receiver)
N_obs_noise     = out[3]  # Independent observations

#__ Noise apparent slowness [s.m-1] __
synthetics = np.load('../data/SYNTHETICS.npz')
s0_J       = synthetics['s0'][0]

Set inversion parameters...

In [7]:
#__ Signals present __
sigs = [
    'radial2D',                 # Alternative parameterisation
    dict(
        sig       = 'noise2D',  # Cylindrically-harmonic noise field
        s         = s0_J,       # Local apparent slowness [s.m-1]
        max_order = 1           # Maximum cylindrical-harmonic order
    )
]

#__ Prior bounds __
lower = np.array([                  # Lower bound
    50.,  -5.0e-4, 8.0e-5, 1.0e-6,  #     'radial2D' parameters
    50., -0.8, -0.8                 #     'noise2D' parameters
])
upper = np.array([                  # Upper bound
    100., -4.2e-4, 1.6e-4, 5.0e-5,  #     'radial2D' parameters
    100., 0.8,  0.8                 #     'noise2D parameters
])

#__ Number of flows __
N_flow = 8

#__ Inversion parameters __
learning_rate = 5e-4                       # Learning rate for the optimiser
N_iterations  = 30000                      # Number of iterations for optimiser
N_montecarlo  = 100                        # Number of samples to estimate ELBO

#__ Number of samples to draw __
N_samples = 20000

Use `topophone2.topophone.decompose` helper function to run full workflow...

In [8]:
#__ Perform inversion __
decompose(
    Gamma_obs       = Gamma_obs,
    N_obs           = N_obs,
    Gamma_obs_noise = Gamma_obs_noise,
    N_obs_noise     = N_obs_noise,
    x               = x,
    freq            = freq,
    sigs            = sigs,
    custom_cf       = custom_cf,
    prior_bounds    = (lower,upper),
    N_flow          = N_flow,
    learning_rate   = learning_rate,
    outpath         = f"{model_dir}",
    N_iterations    = N_iterations,
    N_montecarlo    = N_montecarlo,
    N_out           = 6,
    verbose         = True,
    N_samples       = N_samples,
    calc_likelihood = True
)

#__ Read results __
m0      = np.loadtxt(f"{model_dir}/model-samples_iteration-{N_iterations:05d}.txt")
lnp_dm0 = np.loadtxt(f"{model_dir}/log-likelihood_iteration-{N_iterations:05d}.txt")

----------------------------------------

Iteration:     0,	Loss: 1381304.35
The elapsed time is: 0.18 s
Iteration:  5000,	Loss:   14517.61
The elapsed time is: 72.60 s
Iteration: 10000,	Loss:   14315.29
The elapsed time is: 143.60 s
Iteration: 15000,	Loss:   14310.57
The elapsed time is: 218.55 s
Iteration: 20000,	Loss:   14311.10
The elapsed time is: 295.01 s
Iteration: 25000,	Loss:   14310.55
The elapsed time is: 356.79 s
Iteration: 30000,	Loss:   14310.54
The elapsed time is: 432.25 s
----------------------------------------

Finish training!


### 1.3.3 Calculate Bayes' factor between null and alternative parameterisations

First, load results from Example 1.1.1...

In [9]:
#__ Read results __
result1 = np.load('../models/TRIAD_15DB/inversion_results.npz')
m1      = result1['m']
lnp_dm1 = result1['lnp_dm']
bounds1 = result1['prior_bounds']

Then calculate the evidence

In [10]:
#__ Instantiate stats objects __
stats0 = MarginalStats(m=m0, lnp_dm=lnp_dm0, prior_bounds=(lower,upper))
stats1 = MarginalStats(m=m1, lnp_dm=lnp_dm1, prior_bounds=bounds1)

#__ Calculate evidence __
lnp_d0, dlnp_d0 = stats0.calc_evidence()
lnp_d1, dlnp_d1 = stats1.calc_evidence()

The Bayes' factor is the ratio of evidence...

In [11]:
#__ Calculate Bayes' factor __
K  = lnp_d1 - lnp_d0
dK = np.sqrt(np.mean(dlnp_d0)**2 + np.mean(dlnp_d0)**2)
print(f"The Bayes' factor is ({K:.1f} +/- {dK:.1f}) nats.")

The Bayes' factor is (464.8 +/- 0.3) nats.


This result can be compared to the thresholds given by Kass & Raftery [1995]...

| $K$ | $\log_{10} K$ [dits] | $\ln K$ [nats] | Strength of evidence |
|---|---|---|---|
| 1.0 - 3.2 | 0.0 - 0.5 | 0.0 - 1.2 | Not worth more than a bare mention |
| 3.2 - 10.0 | 0.5 - 1 | 1.2 - 2.3 | Substantial |
| 10.0 - 100.0 | 1.0 - 2.0 | 2.3 - 4.6 | Strong |
| > 100.0 | > 2.0 | > 4.6 | Decisive |